# Filterdiagnostik für Transaktionsdaten

Dieses Notebook vergleicht die ungefilterten Transaktionen in `data/interim/transactions_per_year` mit dem gefilterten Output in `data/interim/transactions_per_year_filtered` (<- hier noch separate Transaktionen).

Es zeigt die Anzahl der Reihen, Produkte und Filialen, den Effekt der einzelnen Filter und die aktivsten Produkte, Filialen, Mandanten und Warengruppen im gefilterten Datensatz. Externe Produkte, die weder als FCM noch als Pseudo klassifiziert sind, werden ausgeschlossen.


## Setup

Die Filterbedingungen werden direkt aus `src.data.cleaning.rules` importiert, damit die Auswertung dieselbe Logik wie die Pipeline verwendet.


In [1]:
from pathlib import Path
import sys

import duckdb
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.common import read_parquet_expr
from src.data.cleaning.rules import (
    ALLOWED_FCM_ARTICLE_IDS,
    EXTERNAL_PRODUCT_RULE,
    FCM_RULE,
    MIN_UMS_MENGE,
    PSEUDO_ARTICLE_IDS,
    WEIGHT_CONTENT_LIKE,
    WEIGHT_RULE,
    fcm_filter_condition,
    fcm_or_pseudo_filter_condition,
    transaction_filter_condition,
    ums_menge_filter_condition,
    weight_filter_condition,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 50)

IN_DIR = ROOT / "data" / "interim" / "transactions_per_year"
OUT_DIR = ROOT / "data" / "interim" / "transactions_per_year_filtered"
IN_GLOB = IN_DIR / "transactions_year_*.parquet"
OUT_GLOB = OUT_DIR / "transactions_year_*.parquet"

if not list(IN_DIR.glob("transactions_year_*.parquet")):
    raise FileNotFoundError(f"Keine Parquet-Dateien in {IN_DIR} gefunden")
if not list(OUT_DIR.glob("transactions_year_*.parquet")):
    raise FileNotFoundError(f"Keine Parquet-Dateien in {OUT_DIR} gefunden")

con = duckdb.connect()
con.execute("PRAGMA threads=8")
con.execute("SET preserve_insertion_order=false")

raw_expr = read_parquet_expr(IN_GLOB)
filtered_expr = read_parquet_expr(OUT_GLOB)

print(f"IN_DIR:  {IN_DIR}")
print(f"OUT_DIR: {OUT_DIR}")


IN_DIR:  /Users/vlada/UNI/SoSe2026/ba/ba_code/data/interim/transactions_per_year
OUT_DIR: /Users/vlada/UNI/SoSe2026/ba/ba_code/data/interim/transactions_per_year_filtered


## Filterstatus und technische Bedingungen

Diese Tabelle zeigt, welche Filter aktiv sind und welche konkrete SQL-Bedingung aus der Pipeline verwendet wird. Die fachliche Bedeutung der Filter steht separat unter der Tabelle.


In [2]:
filter_definitions = pd.DataFrame([
    {
        "Filter": "UMS_MENGE",
        "aktiv": True,
        "Bedingung": ums_menge_filter_condition(),
        "Parameter": f"MIN_UMS_MENGE = {MIN_UMS_MENGE}",
    },
    {
        "Filter": "FCM-Artikel",
        "aktiv": FCM_RULE,
        "Bedingung": fcm_filter_condition() if FCM_RULE else "deaktiviert",
        "Parameter": f"{len(ALLOWED_FCM_ARTICLE_IDS):,} erlaubte Artikel-IDs" if FCM_RULE else "-",
    },
    {
        "Filter": "Externe Artikel",
        "aktiv": EXTERNAL_PRODUCT_RULE,
        "Bedingung": fcm_or_pseudo_filter_condition() if EXTERNAL_PRODUCT_RULE else "deaktiviert",
        "Parameter": (
            f"{len(ALLOWED_FCM_ARTICLE_IDS | PSEUDO_ARTICLE_IDS):,} FCM-/Pseudo-Artikel-IDs"
            if EXTERNAL_PRODUCT_RULE else "-"
        ),
    },
    {
        "Filter": "ARTIKEL_INHALT Gewicht",
        "aktiv": WEIGHT_RULE,
        "Bedingung": weight_filter_condition() if WEIGHT_RULE else "deaktiviert",
        "Parameter": f"ARTIKEL_INHALT LIKE {WEIGHT_CONTENT_LIKE!r}" if WEIGHT_RULE else "-",
    },
])

filter_definitions


,Filter,aktiv,Bedingung,Parameter
0,UMS_MENGE,True,"""UMS_MENGE"" > 0.01",MIN_UMS_MENGE = 0.01
1,FCM-Artikel,False,deaktiviert,-
2,Externe Artikel,True,"((""ARTIKEL_ID"" IN (560039, 579781, 1376569, 13...","3,685 FCM-/Pseudo-Artikel-IDs"
3,ARTIKEL_INHALT Gewicht,True,"(CASE WHEN LOWER(COALESCE(CAST(""ARTIKEL_INHALT...",ARTIKEL_INHALT LIKE '%amm%'


## Fachliche Bedeutung der Filter

- `UMS_MENGE`: Behält nur Transaktionszeilen mit einer relevanten positiven Verkaufsmenge. Dadurch werden Nullmengen, Kleinstmengen unterhalb des Schwellwerts und negative Mengen entfernt.
- `FCM-Artikel`: Beschränkt die Daten auf die definierte FCM-Artikelliste und damit auf die untersuchte Produktauswahl.
- `Externe Artikel`: Behält ausschließlich Produkte aus der FCM- oder Pseudo-Artikelliste. Produkte, die zu keiner der beiden Listen gehören, werden vollständig aus dem Datensatz entfernt.
- `ARTIKEL_INHALT Gewicht`: Behält Produkte, bei denen eine Abverkaufsmenge in kg ableitbar ist. Die Filterregel prüft dafür die Inhaltsangabe `ARTIKEL_INHALT` auf Gramm-/Kilogramm-Angaben. Im gefilterten Output wird `ABVERKAUFTE_MENGE_KG` anschließend aus `GRAMM_BON` gebildet: Für echte Gewichtsartikel (`GEWICHTSARTIKEL = 1`) wird `GRAMM_BON` direkt verwendet; bei verpackten Artikeln wird `GRAMM_BON` nur verwendet, wenn `ARTIKEL_INHALT` eine Gewichtseinheit wie `kg`, `Kilogramm`, `g`, `gr.` oder `Gramm` enthält. Für andere Artikel wäre keine belastbare kg-Menge ableitbar.


## Umfang vor und nach dem Filtern

Eine Reihe ist hier eine eindeutige Kombination aus `ARTIKEL_ID` und `MARKT_ID`.


In [3]:
def where_clause(condition):
    return f"WHERE {condition}" if condition else ""


def count_snapshot(expr, condition=None):
    where_sql = where_clause(condition)
    return con.execute(
        f"""
        SELECT
            COUNT(*)::BIGINT AS Zeilen,
            COUNT(*) FILTER (WHERE ARTIKEL_ID IS NOT NULL AND MARKT_ID IS NOT NULL)::BIGINT AS Zeilen_mit_Reihenschluessel,
            COUNT(DISTINCT (ARTIKEL_ID, MARKT_ID))::BIGINT AS Reihen,
            COUNT(DISTINCT ARTIKEL_ID)::BIGINT AS Produkte,
            COUNT(DISTINCT MARKT_ID)::BIGINT AS Filialen,
            MIN(CAST(DATE AS DATE)) AS erster_Tag,
            MAX(CAST(DATE AS DATE)) AS letzter_Tag
        FROM {expr}
        {where_sql}
        """
    ).fetchdf().iloc[0].to_dict()


overview = pd.DataFrame(
    [
        {"Datensatz": "ungefiltert", **count_snapshot(raw_expr)},
        {"Datensatz": "gefiltert", **count_snapshot(filtered_expr)},
    ]
)
overview["entfernte_Zeilen"] = overview["Zeilen"].iloc[0] - overview["Zeilen"]
overview["behaltener_Zeilenanteil_%"] = (100 * overview["Zeilen"] / overview["Zeilen"].iloc[0]).round(2)

overview


,Datensatz,Zeilen,Zeilen_mit_Reihenschluessel,Reihen,Produkte,Filialen,erster_Tag,letzter_Tag,entfernte_Zeilen,behaltener_Zeilenanteil_%
0,ungefiltert,65521921,65521921,151666,2350,221,2023-07-22,2026-07-21,0,100.00
1,gefiltert,32424401,32424401,66272,681,213,2023-07-22,2026-07-21,33097520,49.49


## Effekt der einzelnen Filter

Die Filter werden sequenziell ausgewertet. `entfernt_*` bedeutet daher: zusätzlich entfernt nach allen vorherigen Filtern.


In [4]:
def combine_conditions(left, right):
    return f"({left}) AND ({right})" if left else right


filter_steps = [("UMS_MENGE", ums_menge_filter_condition())]
if FCM_RULE:
    filter_steps.append(("FCM-Artikel", fcm_filter_condition()))
if EXTERNAL_PRODUCT_RULE:
    filter_steps.append(("Externe Artikel", fcm_or_pseudo_filter_condition()))
if WEIGHT_RULE:
    filter_steps.append(("ARTIKEL_INHALT Gewicht", weight_filter_condition()))

impact_rows = []
current_condition = None
previous = count_snapshot(raw_expr)

for filter_name, filter_condition in filter_steps:
    current_condition = combine_conditions(current_condition, filter_condition)
    current = count_snapshot(raw_expr, current_condition)
    row = {
        "Filter": filter_name,
        "vorher_Zeilen": previous["Zeilen"],
        "nachher_Zeilen": current["Zeilen"],
        "entfernt_Zeilen": previous["Zeilen"] - current["Zeilen"],
        "entfernt_Zeilen_%": round(
            100 * (previous["Zeilen"] - current["Zeilen"]) / previous["Zeilen"], 2
        ) if previous["Zeilen"] else 0,
        "vorher_Reihen": previous["Reihen"],
        "nachher_Reihen": current["Reihen"],
        "entfernt_Reihen": previous["Reihen"] - current["Reihen"],
        "vorher_Produkte": previous["Produkte"],
        "nachher_Produkte": current["Produkte"],
        "entfernt_Produkte": previous["Produkte"] - current["Produkte"],
        "vorher_Filialen": previous["Filialen"],
        "nachher_Filialen": current["Filialen"],
        "entfernt_Filialen": previous["Filialen"] - current["Filialen"],
    }
    impact_rows.append(row)
    previous = current

filter_impact = pd.DataFrame(impact_rows)
expected_kept = count_snapshot(raw_expr, transaction_filter_condition())
actual_kept = count_snapshot(filtered_expr)

if expected_kept["Zeilen"] != actual_kept["Zeilen"]:
    print(
        "WARNUNG: Die erwartete Zeilenzahl nach Filterlogik weicht vom vorhandenen OUT_DIR ab: "
        f"erwartet={expected_kept['Zeilen']:,}, OUT_DIR={actual_kept['Zeilen']:,}"
    )

filter_impact


,Filter,vorher_Zeilen,nachher_Zeilen,entfernt_Zeilen,entfernt_Zeilen_%,vorher_Reihen,nachher_Reihen,entfernt_Reihen,vorher_Produkte,nachher_Produkte,entfernt_Produkte,vorher_Filialen,nachher_Filialen,entfernt_Filialen
0,UMS_MENGE,65521921,65513907,8014,0.01,151666,151614,52,2350,2350,0,221,221,0
1,Externe Artikel,65513907,37448276,28065631,42.84,151614,82115,69499,2350,850,1500,221,216,5
2,ARTIKEL_INHALT Gewicht,37448276,36886208,562068,1.50,82115,77175,4940,850,772,78,216,213,3


## Aktivste Entitäten im gefilterten Datensatz

Aktivität wird für Produkte, Filialen, Mandanten und Warengruppen über die Anzahl der Transaktionszeilen gemessen. Ergänzend werden Tage, Produkte, Filialen, Menge und Umsatz ausgewiesen, soweit sie für die jeweilige Entität sinnvoll sind.


In [5]:
def top_entities(group_cols, label_cols=None, top_n=10, order_by="Zeilen DESC"):
    label_cols = label_cols or []
    select_cols = []
    group_sql = []
    for col in group_cols:
        select_cols.append(col)
        group_sql.append(col)
    for col in label_cols:
        select_cols.append(f"arg_max({col}, DATE) AS {col}")

    select_sql = ",\n            ".join(select_cols)
    group_by_sql = ", ".join(group_sql)

    return con.execute(
        f"""
        SELECT
            {select_sql},
            COUNT(*)::BIGINT AS Zeilen,
            COUNT(DISTINCT CAST(DATE AS DATE))::BIGINT AS Nachfragetage,
            COUNT(DISTINCT ARTIKEL_ID)::BIGINT AS Produkte,
            COUNT(DISTINCT MARKT_ID)::BIGINT AS Filialen,
            SUM(COALESCE(UMS_MENGE, 0.0))::DOUBLE AS Summe_UMS_MENGE,
            SUM(COALESCE(UMS_VK_WERT, 0.0))::DOUBLE AS Summe_UMS_VK_WERT
        FROM {filtered_expr}
        GROUP BY {group_by_sql}
        ORDER BY {order_by}
        LIMIT {top_n}
        """
    ).fetchdf()


def display_top(title, dataframe):
    print(title)
    display(dataframe)


In [6]:
top_products = top_entities(
    group_cols=["ARTIKEL_ID"],
    label_cols=["ARTIKEL_BEZ", "ARTIKEL_INHALT", "VERKAUFSEINHEIT", "GEWICHTSARTIKEL", "WGR_ID", "N_WARENKLASSE_KBEZ"],
    order_by="Zeilen DESC",
)
display_top("Aktivste Produkte nach Transaktionszeilen", top_products)


Aktivste Produkte nach Transaktionszeilen


,ARTIKEL_ID,ARTIKEL_BEZ,ARTIKEL_INHALT,VERKAUFSEINHEIT,GEWICHTSARTIKEL,WGR_ID,N_WARENKLASSE_KBEZ,Zeilen,Nachfragetage,Produkte,Filialen,Summe_UMS_MENGE,Summe_UMS_VK_WERT
0,316623,Hackfleisch gemischt Schwein und Rind,1 Kilogramm,kg,1,890,Schweinefleisch,3561280,1051,1,202,2.782773e+06,2.460206e+07
1,316629,Thüringer Mett vom Schwein,1 Kilogramm,kg,1,890,Schweinefleisch,1686043,1026,1,199,7.863136e+05,6.229680e+06
2,316627,Hackfleisch vom Rind,1 Kilogramm,kg,1,890,Rindfleisch,1663036,1049,1,201,1.049595e+06,1.237873e+07
3,317123,"Nackensteak mariniert, vom Schwein",1 Kilogramm,kg,1,890,Schweinefleisch,1401279,1043,1,198,1.044127e+06,9.682073e+06
4,316638,Schinkenmett gewürzt vom Schwein,1 Kilogramm,kg,1,890,Schweinefleisch,1395124,1024,1,199,5.971209e+05,5.265147e+06
5,317045,"Gyrospfanne, Schinkenfleisch in Streifen, gewürzt",1 Kilogramm,kg,1,890,Schweinefleisch,1176721,1040,1,199,1.113540e+06,9.317636e+06
6,316646,Grobe Bratwurst vom Schwein,1 Kilogramm,kg,1,890,Schweinefleisch,1016293,1041,1,199,5.398389e+05,4.786639e+06
7,317119,Schweinerückensteaks mariniert,1 Kilogramm,kg,1,890,Schweinefleisch,907428,1029,1,198,4.641093e+05,4.442068e+06
8,317157,"Bauchscheiben gewürzt, vom Schwein",1 Kilogramm,kg,1,890,Schweinefleisch,816722,1035,1,198,4.748024e+05,3.981402e+06
9,316783,Oberschalenschnitzel vom Schwein,1 Kilogramm,kg,1,890,Schweinefleisch,696283,1030,1,198,5.548645e+05,4.978965e+06


In [7]:
top_stores = top_entities(
    group_cols=["MARKT_ID"],
    label_cols=["MARKT_NR", "MANDANT_ID", "LEH_SEH"],
)
display_top("Aktivste Filialen", top_stores)


Aktivste Filialen


,MARKT_ID,MARKT_NR,MANDANT_ID,LEH_SEH,Zeilen,Nachfragetage,Produkte,Filialen,Summe_UMS_MENGE,Summe_UMS_VK_WERT
0,1100016,16,110,LEH,481316,911,376,1,338713.1669,3.400970e+06
1,1300021,21,130,LEH,472298,909,392,1,389272.7528,3.667028e+06
2,1300023,23,130,LEH,415184,909,387,1,331922.3028,3.150095e+06
3,1300005,5,130,LEH,410825,909,399,1,330290.7299,3.124755e+06
4,1300009,9,130,LEH,401765,906,359,1,340015.4645,3.253245e+06
5,1300001,1,130,LEH,400682,909,477,1,277203.3774,2.978993e+06
6,1300024,24,130,LEH,368676,909,426,1,264695.1575,2.676059e+06
7,1300006,6,130,LEH,357462,910,377,1,273837.7297,2.643950e+06
8,1300004,4,130,LEH,352617,909,408,1,287049.9504,2.911199e+06
9,1300020,20,130,LEH,344412,908,397,1,257239.1131,2.587273e+06


In [8]:
top_mandants = top_entities(
    group_cols=["MANDANT_ID"],
    label_cols=["LEH_SEH"],
)
display_top("Aktivste Mandanten", top_mandants)


Aktivste Mandanten


,MANDANT_ID,LEH_SEH,Zeilen,Nachfragetage,Produkte,Filialen,Summe_UMS_MENGE,Summe_UMS_VK_WERT
0,110,LEH,12156386,1049,511,66,9.025724e+06,8.561269e+07
1,130,LEH,6670704,916,568,20,5.258698e+06,5.139220e+07
2,125,LEH,6208773,1009,478,62,3.800938e+06,3.829000e+07
3,120,LEH,3520595,931,475,31,2.688764e+06,2.493906e+07
4,135,LEH,2880203,912,473,26,1.899743e+06,1.920731e+07
5,115,LEH,987740,913,418,8,6.547097e+05,6.296547e+06


In [9]:
top_warengruppen = top_entities(
    group_cols=["WGR_ID"],
    label_cols=["N_WARENKLASSE_KBEZ"],
)
display_top("Aktivste Warengruppen", top_warengruppen)


Aktivste Warengruppen


,WGR_ID,N_WARENKLASSE_KBEZ,Zeilen,Nachfragetage,Produkte,Filialen,Summe_UMS_MENGE,Summe_UMS_VK_WERT
0,890,Schweinefleisch,31499392,1058,576,211,2.284845e+07,2.231961e+08
1,900,Fleischkäseprodukte,925009,980,105,203,4.801220e+05,2.541708e+06


## Hinweise

Die Filterwirkung ist sequenziell berechnet. Wenn ein Datensatz bereits durch einen früheren Filter entfernt wurde, wird er bei späteren Filtern nicht erneut gezählt.
